할 것:
- dropna → 714개 확인
- stratify=y로 분리 (불균형 비율 유지)
- confusion_matrix 출력
- classification_report 출력 (target_names 설정)
- AUC 계산 → predict_proba[:, 1] 사용
- "무조건 사망 예측하면 정확도가 얼마인지" 직접 계산



타이타닉 노트북 — confusion_matrix, classification_report, AUC 출력 셀 포함. SGD 에포크 그래프 1개 포함 (train/test accuracy, x축 epoch, y축 accuracy).

In [34]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

In [35]:
# Step 2. 데이터 불러오기 + 전처리
titanic = sns.load_dataset('titanic')
titanic_clean = titanic[['survived', 'pclass', 'sex',
                          'age', 'sibsp', 'parch', 'fare']].dropna()
print("전처리 후 샘플 수:", len(titanic_clean))

전처리 후 샘플 수: 714


In [36]:
print(titanic_clean.columns)
print(titanic_clean.head())

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare'], dtype='str')
   survived  pclass     sex   age  sibsp  parch     fare
0         0       3    male  22.0      1      0   7.2500
1         1       1  female  38.0      1      0  71.2833
2         1       3  female  26.0      0      0   7.9250
3         1       1  female  35.0      1      0  53.1000
4         0       3    male  35.0      0      0   8.0500


In [37]:
# Step 3. 인코딩
# male → 0, female → 1
titanic_clean['sex'] = titanic_clean['sex'].map({'male': 0, 'female': 1})

print(titanic_clean.head())

   survived  pclass  sex   age  sibsp  parch     fare
0         0       3    0  22.0      1      0   7.2500
1         1       1    1  38.0      1      0  71.2833
2         1       3    1  26.0      0      0   7.9250
3         1       1    1  35.0      1      0  53.1000
4         0       3    0  35.0      0      0   8.0500


In [38]:
# Step 4. 문제지 / 정답지 분리
X = titanic_clean.drop('survived', axis=1).values
y = titanic_clean['survived'].values

print("생존:", y.sum(), "명")
print("사망:", len(y) - y.sum(), "명")
print("생존 비율:", y.mean().round(3))

생존: 290 명
사망: 424 명
생존 비율: 0.406


In [39]:
# Step 5. 훈련/테스트 분리

X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, stratify=y
)
print("훈련:", X_train.shape)
print("테스트:", X_test.shape)

훈련: (535, 6)
테스트: (179, 6)


In [40]:
# Step 6. 스케일링
ss = StandardScaler()
X_train_s = ss.fit_transform(X_train)  # 훈련만 fit
X_test_s  = ss.transform(X_test)       # 테스트는 transform만

In [48]:
# Step 7. 로지스틱 회귀 학습
lr = LogisticRegression()
lr.fit(X_train_s, y_train)
y_pred = lr.predict(X_test_s)

print("정확도:", round(lr.score(X_train_s, y_train), 3))
print("정확도:", round(lr.score(X_test_s, y_test), 3))

정확도: 0.804
정확도: 0.793


In [50]:
# Step 8. predict_proba DataFrame 출력 (제출 항목)
# ──────────────────────────────────────────────────────────────
 
# [역할] 각 승객의 사망/생존 확률을 표로 출력
# [원리] predict_proba → [[사망확률, 생존확률], ...]
#        .round(3) → 소수점 3자리
# [기본형] pd.DataFrame({'열이름': 값, ...})
proba_df = pd.DataFrame({
    '사망 확률': lr.predict_proba(X_test_s)[:, 0].round(3),
    '생존 확률': lr.predict_proba(X_test_s)[:, 1].round(3),
    '실제 정답': y_test,
    '예측':      y_pred,
    '결과':      ['✅' if a == p else '❌' 
                  for a, p in zip(y_test, y_pred)]  # 맞으면 ✅ 틀리면 ❌
})
print(proba_df.head(10))

# Step 8. 혼동행렬
cm = confusion_matrix(y_test, y_pred)
print("혼동행렬:")
print(cm)

TN = cm[0][0]
FP = cm[0][1]
FN = cm[1][0]
TP = cm[1][1]

print(f"\nTN (사망→사망 ✅): {TN}")
print(f"FP (사망→생존 ❌): {FP}")
print(f"FN (생존→사망 ❌): {FN}")
print(f"TP (생존→생존 ✅): {TP}")

   사망 확률  생존 확률  실제 정답  예측 결과
0  0.911  0.089      0   0  ✅
1  0.797  0.203      0   0  ✅
2  0.909  0.091      1   0  ❌
3  0.902  0.098      0   0  ✅
4  0.902  0.098      0   0  ✅
5  0.454  0.546      0   1  ❌
6  0.893  0.107      0   0  ✅
7  0.076  0.924      1   1  ✅
8  0.594  0.406      0   0  ✅
9  0.540  0.460      0   0  ✅
혼동행렬:
[[87 19]
 [18 55]]

TN (사망→사망 ✅): 87
FP (사망→생존 ❌): 19
FN (생존→사망 ❌): 18
TP (생존→생존 ✅): 55


In [ ]:
# Step 9. 분류보고서
print(classification_report(y_test, y_pred, target_names=['사망(0)', '생존(1)']))

              precision    recall  f1-score   support

       사망(0)       0.83      0.82      0.82       106
       생존(1)       0.74      0.75      0.75        73

    accuracy                           0.79       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.79      0.79      0.79       179



먼저 공식부터
Precision = 내가 맞다고 한 것 중 진짜 맞은 비율
Recall    = 실제로 맞은 것 중 내가 찾아낸 비율
F1        = Precision과 Recall의 평균
Support   = 실제 샘플 수

사망(0) 행
precision 0.83  → 사망이라 예측한 것 중 83%가 실제 사망
recall    0.82  → 실제 사망자 106명 중 82%를 맞힘
f1-score  0.82  → 둘의 평균
support   106   → 실제 사망자 106명

생존(1) 행
precision 0.74  → 생존이라 예측한 것 중 74%가 실제 생존
recall    0.75  → 실제 생존자 73명 중 75%를 맞힘
f1-score  0.75  → 둘의 평균
support   73    → 실제 생존자 73명

accuracy
0.79 → 전체 179명 중 79%를 맞힘

macro avg vs weighted avg
macro avg    → 사망/생존 점수를 단순 평균 (샘플 수 무시)
weighted avg → 샘플 수를 반영한 평균
              (사망 106명 > 생존 73명이라 사망 쪽 가중치 더 높음)

한 줄 요약
사망 예측이 생존 예측보다 더 잘 됩니다.
이유: 사망자(106명)가 생존자(73명)보다 많아서
모델이 사망 패턴을 더 많이 학습했기 때문입니다.

In [44]:
# Step 10. AUC
y_score = lr.predict_proba(X_test_s)[:, 1]
auc = roc_auc_score(y_test, y_score)
print(f"AUC: {auc:.3f}")

AUC: 0.863


In [45]:
# Step 11. 무조건 사망 예측 정확도 (정확도의 함정)
dummy_pred = np.zeros(len(y_test))
dummy_acc  = (dummy_pred == y_test).mean()

print(f"무조건 사망 예측 정확도: {dummy_acc:.3f}")
print(f"로지스틱 회귀 정확도:   {lr.score(X_test_s, y_test):.3f}")
print(f"차이: {lr.score(X_test_s, y_test) - dummy_acc:.3f}")


무조건 사망 예측 정확도: 0.592
로지스틱 회귀 정확도:   0.793
차이: 0.201


In [46]:
# Step 12. 임계값 0.3 적용
# ══════════════════════════════════════════════════════════════
y_pred_30 = (y_score >= 0.3).astype(int)
print("임계값 0.3 적용 후 분류 보고서:")
print(classification_report(y_test, y_pred_30,
                             target_names=['사망(0)', '생존(1)']))

임계값 0.3 적용 후 분류 보고서:
              precision    recall  f1-score   support

       사망(0)       0.85      0.69      0.76       106
       생존(1)       0.65      0.82      0.72        73

    accuracy                           0.74       179
   macro avg       0.75      0.76      0.74       179
weighted avg       0.77      0.74      0.75       179



전체 흐름
타이타닉 데이터로 생존 여부를 예측하는 분류 모델을 만드는 실습입니다.
Step 1 에서는 필요한 도구들을 전부 불러왔습니다. pandas, numpy, seaborn, sklearn의 여러 함수들입니다.

Step 2 에서는 seaborn에 내장된 타이타닉 데이터를 불러왔습니다. 891명 중 필요한 7개 열만 골라서 빈 칸이 있는 행을 제거했습니다. 결과적으로 714명이 남았습니다.

Step 3 에서는 sex 열의 male과 female을 0과 1로 바꿨습니다. 모델은 문자를 읽지 못하기 때문입니다.

Step 4 에서는 문제지와 정답지를 분리했습니다. survived 열을 제외한 나머지가 문제지(X), survived 열이 정답지(y)입니다. 생존자는 290명(40.6%), 사망자는 424명(59.4%)으로 불균형 데이터입니다.

Step 5 에서는 데이터를 훈련용 535개, 테스트용 179개로 나눴습니다. stratify=y를 써서 생존:사망 비율을 유지하면서 나눴습니다.

Step 6 에서는 스케일링을 했습니다. 훈련 데이터로만 fit_transform하고 테스트는 transform만 했습니다. Data Leakage 방지입니다.

Step 7 에서는 로지스틱 회귀 모델을 학습시켰습니다.

Step 8 에서는 predict_proba 결과를 DataFrame으로 출력합니다. 각 승객의 사망/생존 확률을 표로 볼 수 있으며 혼동행렬을 출력했습니다. TN=87, FP=19, FN=18, TP=55가 나왔습니다. 사망자 106명 중 87명을 맞히고 19명을 틀렸습니다. 생존자 73명 중 55명을 맞히고 18명을 놓쳤습니다.

Step 9 에서는 분류 보고서를 출력했습니다. 사망 예측 정밀도 83%, 생존 예측 정밀도 74%, 전체 정확도 79%가 나왔습니다.

Step 10 에서는 AUC를 계산했습니다. 0.863으로 기대값 0.864에 거의 일치합니다. 모델이 생존/사망을 86% 수준으로 잘 구분한다는 뜻입니다.

Step 11 에서는 무조건 사망으로 예측했을 때 정확도를 계산했습니다. 59.2%가 나왔습니다. 로지스틱 회귀는 79.3%라서 20.1% 차이가 납니다. 이게 정확도의 함정입니다. 아무 모델 없이 전부 사망이라고 해도 59%가 나오기 때문에 정확도만으로 모델을 평가하면 안 됩니다.

Step 12 에서는 임계값을 0.5에서 0.3으로 낮췄습니다. 생존 Recall이 75%에서 82%로 올라갔습니다. 더 많은 사람을 생존으로 예측하니까 실제 생존자를 더 많이 맞힌 겁니다. 대신 Precision이 74%에서 65%로 떨어졌습니다. FP가 늘어난 겁니다.

Step 13 에서는 임계값을 0.3으로 낮춰서 생존 Recall이 올라가는 것을 확인합니다.

Step 14 에서는 SGD 300 에포크 그래프를 그려서 조기 종료 시점을 찾습니다.